In [ ]:
import time
import win32clipboard
import win32gui
import win32process
import psutil
import pyperclip

def get_active_window_info():
    """取得當前最前景視窗的標題與執行檔名稱"""
    hwnd = win32gui.GetForegroundWindow()
    if not hwnd:
        return "未知視窗", "未知程式"
    
    title = win32gui.GetWindowText(hwnd)
    try:
        _, pid = win32process.GetWindowThreadProcessId(hwnd)
        proc_name = psutil.Process(pid).name()
    except Exception:
        proc_name = "未知程式"
        
    return title, proc_name

def get_copied_files():
    """嘗試從剪貼簿讀取檔案清單 (CF_HDROP 格式)"""
    copied_files = []
    try:
        win32clipboard.OpenClipboard()
        # CF_HDROP (15) 是 Windows 代表檔案清單的剪貼簿格式
        if win32clipboard.IsClipboardFormatAvailable(win32clipboard.CF_HDROP):
            data = win32clipboard.GetClipboardData(win32clipboard.CF_HDROP)
            if data:
                copied_files = list(data)
    except Exception:
        pass
    finally:
        try:
            win32clipboard.CloseClipboard()
        except Exception:
            pass
    return copied_files

def monitor_clipboard():
    print("剪貼簿監聽中（支援文字與檔案複製，按下 Ctrl+C 結束）...\n" + "="*50)
    
    last_text = ""
    last_files = []

    while True:
        try:
            # 1. 檢查是否有檔案複製 (CF_HDROP)
            current_files = get_copied_files()
            if current_files and current_files != last_files:
                last_files = current_files
                title, proc = get_active_window_info()
                
                print(f"【檢測到檔案複製】")
                print(f"來源程式: {proc}")
                print(f"視窗標題: {title}")
                print(f"複製檔案數量: {len(current_files)}")
                for idx, file_path in enumerate(current_files, 1):
                    print(f"  {idx}. {file_path}")
                print("="*50)

            # 2. 檢查是否有純文字複製 (非檔案時)
            if not current_files:
                current_text = pyperclip.paste()
                if current_text and current_text != last_text:
                    last_text = current_text
                    title, proc = get_active_window_info()
                    
                    print(f"【檢測到文字複製】")
                    print(f"來源程式: {proc}")
                    print(f"視窗標題: {title}")
                    print(f"複製文字: {current_text[:60]}{'...' if len(current_text) > 60 else ''}")
                    print("="*50)

            time.sleep(0.3)  # 輪詢間隔 0.3 秒

        except KeyboardInterrupt:
            print("\n已停止監聽。")
            break
        except Exception as e:
            time.sleep(0.5)

if __name__ == "__main__":
    monitor_clipboard()

剪貼簿監聽中...（按下 Ctrl+C 結束程式）
----------------------------------------

已停止監聽。
